In [ ]:
%pip install datasets transformers torch accelerate

In [1]:
# Datasets
from datasets import load_dataset, concatenate_datasets

# Output
import json
from collections import Counter
import pandas as pd

# Model
import torch
import re
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import time

/common/home/users/c/chinyee.lew.2022/jupyterlab-venv-tf-217/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
dataset = load_dataset("gsm8k", "main", split="test")

# Filter datasets
dataset = dataset.filter(
    lambda x: len(x["answer"]) > 404
)

print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 245
})


In [3]:
# Load model
model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 179.88it/s]


In [4]:
# Helper function
def extract_true_answer(answer):
    return answer.split("####")[-1].strip()

def extract_true_reasoning(answer):
    return answer.split("####")[0].strip()

def count_tokens(input_text, output_text):
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens

In [5]:
def build_prompt(question, true_answer, true_reasoning):

    return f"""
You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step.
1. Analyze the problem and identify key values.
2. Compute sequentially, showing all intermediate steps.
3. Verify your final calculation before concluding.

CRITICAL OUTPUT RULES:
- Return ONLY a valid JSON object.
- Do NOT use markdown, code blocks, or backticks.
- Do NOT include any text, greetings, or explanations outside the JSON.
- Do NOT repeat the question or include ####.
- "model_reasoning": Clear step-by-step logic. Avoid using curly braces {{}} in the text to prevent JSON parsing errors.
- "model_answer": Final numeric value ONLY (e.g., 42, 15.5). No units, words, or symbols.

Required JSON Format:
{{
  "model_reasoning": "...",
  "model_answer": "..."
}}

Problem:
{question}
"""

In [6]:
pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 315.61it/s]


In [7]:
results = []

start_time = time.time()

num_attempts = 2
consistent_count = 0

def extract_json(text):
    matches = re.finditer(r"\{.*?\}", text, flags=re.DOTALL)
    for match in matches:
        try:
            parsed = json.loads(match.group())
            if "model_answer" not in parsed:
                parsed["model_answer"] = ""
            if "model_reasoning" not in parsed:
                parsed["model_reasoning"] = ""
            return parsed
        except json.JSONDecodeError:
            continue
    return {"model_answer": "", "model_reasoning": ""}


for i, example in enumerate(dataset):
    question = example["question"]

    true_answer = extract_true_answer(example["answer"])
    true_reasoning = extract_true_reasoning(example["answer"])

    prompt = build_prompt(
        question=question,
        true_answer=true_answer,
        true_reasoning=true_reasoning
    )

    model_responses = []
    model_answers = []
    model_correct = []
    model_input_tokens = []
    model_output_tokens = []
    model_reasonings = []

    # Run 2 attempts (temp=0.7)
    for attempt in range(num_attempts):

        output = pipe(
            prompt,
            max_new_tokens=320,
            return_full_text=False,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1
        )[0]

        response = output["generated_text"]
        parsed = extract_json(response)

        model_answer = parsed.get("model_answer", "")
        model_reasoning = parsed.get("model_reasoning", "")

        is_correct = (model_answer == true_answer)

        input_tokens, output_tokens = count_tokens(prompt, response)

        # store per attempt
        model_answers.append(model_answer)
        model_correct.append(is_correct)
        model_input_tokens.append(input_tokens)
        model_output_tokens.append(output_tokens)
        model_reasonings.append(model_reasoning)
        model_responses.append(response)

    response_1 = model_responses[0]
    response_2 = model_responses[1]

    model_reasoning = model_reasonings[0]

    # Consistency (between the 2 attempts)
    cleaned_answers = [ans for ans in model_answers]

    is_consistent = (
        len(set(cleaned_answers)) == 1
        and all(ans != "" for ans in cleaned_answers)
    )

    if is_consistent:
        consistent_count += 1

    # Average metrics
    avg_accuracy = sum(model_correct) / num_attempts
    avg_input_tokens = sum(model_input_tokens) / num_attempts
    avg_output_tokens = sum(model_output_tokens) / num_attempts

    results.append({
        "question": question,
        "true_reasoning": true_reasoning,
        "true_answer": true_answer,
        "model_reasoning": model_reasoning,
        
        # per attempt
        "model_answers": model_answers,
        "model_correct": model_correct,
        "model_input_tokens": model_input_tokens,
        "model_output_tokens": model_output_tokens,

        # aggregated
        "avg_accuracy": avg_accuracy,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "consistent": is_consistent,
        "response_1": response_1,
        "response_2": response_2
    })

end_time = time.time()

Passing `generation_config` together with generation-related arguments=({'temperature', 'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/trans

In [8]:
df = pd.DataFrame(results)
# df.to_csv("qwen_gsm8k_results.csv", index=False)

df["true_answer"] = pd.to_numeric(df["true_answer"], errors="coerce")

df.to_json("results.json", orient="records", indent=4)

In [9]:
# Metrics

total_runtime = end_time - start_time
avg_runtime = total_runtime / len(results)
print(f"Total runtime for {len(results)} examples: {total_runtime:.3f} sec")
print(f"Average runtime per example: {avg_runtime:.3f} sec")

accuracy = sum(r["avg_accuracy"] for r in results) / len(results)
print("Final answer accuracy:", accuracy)

avg_input_tokens = sum(r["avg_input_tokens"] for r in results) / len(results)
avg_output_tokens = sum(r["avg_output_tokens"] for r in results) / len(results)

print("Avg input tokens:", avg_input_tokens)
print("Avg output tokens:", avg_output_tokens)

total_questions = len(results)
consistency_score = consistent_count / total_questions if total_questions > 0 else 0
print(f"Consistency score: {consistency_score:.4f}")

Total runtime for 245 examples: 3496.743 sec
Average runtime per example: 14.272 sec
Final answer accuracy: 0.01020408163265306
Avg input tokens: 258.38775510204084
Avg output tokens: 270.99387755102043
Consistency score: 0.0000
